In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymongo import MongoClient
from bson.json_util import dumps
import base64

# Import the CRUD Python module from Project One
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################
# Hardcoded aacuser credentials for the MongoDB connection (the "model" layer of MVC).
username = "aacuser"
password = "threeforty"
shelter = AnimalShelter(username, password)

# Load all records into a dataframe for the initial (unfiltered) table view.
df = pd.DataFrame.from_records(shelter.read({}))

# Drop the MongoDB '_id' column, which is an ObjectID type the data_table cannot render.
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash('SimpleExample')

# Read and base64-encode the Grazioso Salvare logo so it can be embedded in the page.
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Hr(),
    # Grazioso Salvare logo (branding)
    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '150px', 'width': 'auto'}
        )
    ),
    html.Center(html.B(html.H1('SNHU CS-340 Dashboard'))),
    # Unique identifier
    html.Center(html.B(html.H4('Michael Miller'))),
    html.Hr(),

    # Interactive filter options (controller) - radio buttons that drive the widgets.
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin-right': '25px'}
        ),
        style={'textAlign': 'center'}
    ),
    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,
    ),
    html.Br(),
    html.Hr(),

    # Charts row: pie chart (left) and geolocation map (right)
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),
        html.Div(id='map-id', className='col s12 m6')
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Filter callback: the selected radio option runs a CRUD query and updates the table data.
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Each branch uses the CRUD read method to run the matching MongoDB query.
    if filter_type == 'Water':
        dff = pd.DataFrame(list(shelter.read({
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        })))
    elif filter_type == 'Mountain':
        dff = pd.DataFrame(list(shelter.read({
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                              "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        })))
    elif filter_type == 'Disaster':
        dff = pd.DataFrame(list(shelter.read({
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                              "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0}
        })))
    else:
        # Reset: return all records (unfiltered).
        dff = pd.DataFrame.from_records(shelter.read({}))

    # Drop the ObjectID column so the table can render.
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# Highlight a selected column on the data table.
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Pie chart: displays the breed distribution of whatever data is currently in the table.
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None:
        return
    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty or 'breed' not in dff.columns:
        return
    return [
        dcc.Graph(
            figure=px.pie(dff, names='breed', title='Preferred Animals by Breed')
        )
    ]


# Geolocation map: shows the location of the selected row in the table.
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None:
        return
    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return

    # Single row selection: convert the list to a single row index.
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Austin TX is at [30.75, -97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Columns 13/14 are location_lat/location_long; column 4 is breed; column 9 is name.
            dl.Marker(position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                      children=[
                dl.Tooltip(dff.iloc[row, 4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row, 9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode. If port 8050 is taken from a prior
# run, set an alternate port, e.g. app.run_server(port=8051).
app.run_server()

Dash app running on https://pastadrum-cornernina-3000.codio.io/proxy/8050/
